# 🧬 Behavioral Analytics: The Modern Student Triad
### Exploring AI Dependency, Career Anxiety, and Student Burnout

#### 🧠 Project Motivation
Most publicly available education datasets focus strictly on traditional performance metrics like grades, test scores, or attendance records. While useful, they completely miss the massive psychological shifts occurring in the generative AI era. 

This notebook explores the intersection of three highly correlated contemporary phenomena: **Generative AI Dependency**, **Acute Career Anxiety**, and **Chronic Student Burnout**. Rather than existing as isolated issues, these factors form a compounding psychological feedback loop that alters how students learn, cope, and view their professional futures.

#### 🛠️ Dataset Design Philosophy
Unlike simplistic synthetic datasets with perfectly linear relationships, this dataset was intentionally engineered to include real-world dataset characteristics:
* **Overlapping Behaviors:** Students exhibiting high AI usage alongside high manual study efforts.
* **Contradictory Personas:** High-performing students tracking high burnout levels.
* **Controlled Missingness:** Simulating real-world survey drops or unrecorded tracking metrics.
* **Moderate Correlations:** Introducing realistic noise and variance to prevent trivial decision boundaries.

#### Automated Data Ingestion
To ensure maximum reproducibility and maintain clean versioning, we pull the official dataset directly from Kaggle utilizing the `kagglehub` library. 

This automated process fetches the raw behavioral data tracking AI usage metrics, burnout indicators, and career anxiety scales directly into our environment.

* **Dataset Credit:** Sridip Basu via Kaggle (*AI Dependency, Career Anxiety, and Student Burnout Dataset*).

### 📥 Loading the Dataset and Libraries

Before we start, we need to install the necessary Python libraries and **load the dataset**.


In [1]:
# Install the required libraries
%pip install kagglehub pandas numpy matplotlib seaborn scikit-learn -q

%pip install ucimlrepo -q
# Install and update the watermark package to display environment and library version information
%pip install -q -U watermark

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# IMPORT REQUIRED MODULES - Data manipulation and visualization

import os
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Pre-Processing and Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

c:\Users\LarTI\OneDrive\Desktop\Projects\AI_Dependency\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# --- AUTOMATED KAGGLE INGESTION ---
# Download the latest version of the specific student behavioral dataset
path = kagglehub.dataset_download("sridipbasu/ai-depndency-career-anxiety-and-student-burnout")
print("🚀 Path to dataset files:", path)

100%|██████████| 437k/437k [00:00<00:00, 1.03MB/s]

Extracting files...
🚀 Path to dataset files: C:\Users\LarTI\.cache\kagglehub\datasets\sridipbasu\ai-depndency-career-anxiety-and-student-burnout\versions\1


In [4]:
# --- LOCATING AND READING THE CSV ---
# List out all files inside the downloaded repository path to spot the target file
all_files = os.listdir(path)
print("📂 Files discovered in directory:", all_files)

csv_filename = [file for file in all_files if file.endswith('.csv')][0]
full_csv_path = os.path.join(path, csv_filename)

📂 Files discovered in directory: ['ai_dependency_career_anxiety_students.csv', 'dataset_metadata.json', 'feature_dictionary.csv']


In [5]:
# Load the data directly into a primary Pandas DataFrame matrix
df = pd.read_csv(full_csv_path)
print("✅ Data successfully loaded into DataFrame named 'df'!")

✅ Data successfully loaded into DataFrame named 'df'!


In [6]:
# Display the first 5 records to see our column properties and labels
df.head()

,student_id,age,gender,degree_type,stream,year_of_study,college_tier,urban_or_rural,daily_ai_tool_usage_hrs,primary_ai_tools_used,...,daily_study_hours,self_learning_hours_per_week,skill_development_courses_taken,social_media_hrs_per_day,sleep_hours,stress_level,burnout_score,motivation_score,seeks_career_counseling,overall_career_readiness_score
0,STU_00001,25,Male,B.Tech/B.E.,Engineering (Non-CS),3,Tier 3,Rural,0.3,Perplexity,...,3.5,8.7,2,1.5,8.5,1,3,6,0.0,9.09
1,STU_00002,20,Female,B.Tech/B.E.,Engineering (Non-CS),4,Tier 3,Urban,1.9,ChatGPT,...,2.4,8.2,3,4.2,8.3,4,5,3,0.0,6.03
2,STU_00003,25,Female,MBA,CS/IT,1,Tier 3,Urban,3.6,Gemini,...,2.3,16.4,1,2.3,7.0,5,9,3,1.0,6.68
3,STU_00004,23,Male,B.Tech/B.E.,CS/IT,1,Tier 1,Urban,4.1,GitHub Copilot,...,7.7,15.0,3,2.6,7.2,6,6,6,0.0,7.71
4,STU_00005,22,Female,MBA,CS/IT,1,Tier 1,Urban,3.4,ChatGPT,...,2.4,1.1,2,1.9,8.0,3,5,4,0.0,4.64


In [7]:
# It returns the dimensions of the DataFrame as a tuple
df.shape

(15000, 30)

In [8]:
df.sample(15) # Random 15 rows

,student_id,age,gender,degree_type,stream,year_of_study,college_tier,urban_or_rural,daily_ai_tool_usage_hrs,primary_ai_tools_used,...,daily_study_hours,self_learning_hours_per_week,skill_development_courses_taken,social_media_hrs_per_day,sleep_hours,stress_level,burnout_score,motivation_score,seeks_career_counseling,overall_career_readiness_score
4852,STU_04853,21,Female,MBA,Commerce/Management,2,Tier 3,Urban,0.0,NaN,...,2.8,0.0,6,1.6,6.1,5,6,6,0.0,5.61
14735,STU_14736,21,Male,B.Sc/B.A/B.Com,CS/IT,2,Tier 1,Rural,4.8,Claude,...,0.8,5.9,0,3.9,8.4,1,3,5,0.0,3.88
5678,STU_05679,19,Female,B.Tech/B.E.,CS/IT,4,Tier 3,Urban,5.1,Gemini,...,3.9,8.8,2,2.4,7.5,3,6,3,1.0,7.10
8202,STU_08203,22,Female,B.Tech/B.E.,Engineering (Non-CS),1,Tier 3,Urban,1.6,Perplexity,...,1.6,2.2,0,3.4,6.5,5,8,2,0.0,5.65
1669,STU_01670,21,Male,B.Sc/B.A/B.Com,Engineering (Non-CS),2,Tier 3,Urban,0.3,Gemini,...,2.4,7.4,4,2.5,8.2,5,6,6,0.0,6.20
2430,STU_02431,21,Male,B.Sc/B.A/B.Com,Commerce/Management,1,Tier 3,Urban,1.0,ChatGPT,...,3.3,5.8,2,1.7,7.6,6,6,3,1.0,2.00
13430,STU_13431,23,Female,B.Tech/B.E.,Commerce/Management,3,Tier 3,Urban,0.0,NaN,...,0.7,0.0,4,2.4,7.4,4,3,2,0.0,5.52
5863,STU_05864,20,Female,B.Sc/B.A/B.Com,Commerce/Management,1,Tier 2,Rural,0.2,ChatGPT,...,0.0,0.0,0,1.4,5.6,7,7,2,0.0,2.97
5505,STU_05506,23,Non-binary,B.Sc/B.A/B.Com,Commerce/Management,3,Tier 2,Urban,0.0,NaN,...,4.4,3.7,3,2.6,6.9,4,4,6,0.0,5.96
1371,STU_01372,20,Female,B.Sc/B.A/B.Com,CS/IT,2,Tier 3,Rural,4.4,ChatGPT,...,5.0,19.8,3,3.3,6.8,5,5,5,0.0,8.03


In [9]:
df.tail() #Displays the last 5 rows of the DataFrame df.

,student_id,age,gender,degree_type,stream,year_of_study,college_tier,urban_or_rural,daily_ai_tool_usage_hrs,primary_ai_tools_used,...,daily_study_hours,self_learning_hours_per_week,skill_development_courses_taken,social_media_hrs_per_day,sleep_hours,stress_level,burnout_score,motivation_score,seeks_career_counseling,overall_career_readiness_score
14995,STU_14996,21,Female,B.Sc/B.A/B.Com,CS/IT,1,Tier 3,Urban,2.8,ChatGPT,...,0.0,6.2,2,0.2,9.6,4,6,2,0.0,7.02
14996,STU_14997,22,Male,B.Tech/B.E.,Commerce/Management,4,Tier 1,Urban,3.6,ChatGPT,...,6.1,11.7,3,2.9,5.8,5,3,9,1.0,7.31
14997,STU_14998,22,Male,B.Tech/B.E.,CS/IT,2,Tier 1,Rural,1.7,Gemini,...,6.3,17.2,5,0.6,7.4,3,1,9,0.0,8.65
14998,STU_14999,22,Female,MBA,CS/IT,2,Tier 2,Urban,0.0,NaN,...,0.8,0.0,1,2.0,9.1,5,6,2,0.0,4.97
14999,STU_15000,23,Male,M.Tech/M.Sc,Engineering (Non-CS),2,Tier 3,Rural,2.1,ChatGPT,...,2.9,1.0,3,3.9,7.4,7,7,4,0.0,5.28


#### 🔎📊 Exploratory Data Analysis (EDA)

The investigation phase Here, we dive into the data to uncover patterns, anomalies, and valuable insights 💡. We use graphs 📈 and statistical measures to better understand the distribution and characteristics of the data before making any modifications 🛠️.

In [10]:
# Information about the dataframe
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   student_id                       15000 non-null  str    
 1   age                              15000 non-null  int64  
 2   gender                           15000 non-null  str    
 3   degree_type                      15000 non-null  str    
 4   stream                           15000 non-null  str    
 5   year_of_study                    15000 non-null  int64  
 6   college_tier                     15000 non-null  str    
 7   urban_or_rural                   15000 non-null  str    
 8   daily_ai_tool_usage_hrs          15000 non-null  float64
 9   primary_ai_tools_used            11785 non-null  str    
 10  uses_ai_for_assignments          15000 non-null  str    
 11  ai_replaces_own_thinking_score   15000 non-null  int64  
 12  ai_dependency_score          

In [11]:
df.describe(include='all').T  # Generates descriptive statistics for all numeric columns in your DataFrame.

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
student_id,15000,15000,STU_00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,15000.0,NaN,NaN,NaN,21.384533,2.005626,18.0,20.0,21.0,22.0,28.0
gender,15000,3,Male,7840,NaN,NaN,NaN,NaN,NaN,NaN,NaN
degree_type,15000,4,B.Tech/B.E.,6700,NaN,NaN,NaN,NaN,NaN,NaN,NaN
stream,15000,4,CS/IT,6091,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year_of_study,15000.0,NaN,NaN,NaN,2.100067,0.98765,1.0,1.0,2.0,3.0,4.0
college_tier,15000,3,Tier 3,7475,NaN,NaN,NaN,NaN,NaN,NaN,NaN
urban_or_rural,15000,2,Urban,10282,NaN,NaN,NaN,NaN,NaN,NaN,NaN
daily_ai_tool_usage_hrs,15000.0,NaN,NaN,NaN,2.013893,1.735977,0.0,0.4,1.8,3.2,8.0
primary_ai_tools_used,11785,5,ChatGPT,5469,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 🧹 Data Preprocessing & Hygiene
Now that the data is loaded, we begin our cleaning pipeline. 
Our tasks in this phase are:
1. **Find the missing data:** Using the command **df.isna().sum()** 
2. **Drop Identifiers:** Remove `student_id` since it has no predictive or statistical power.
3. **Handle Missing Data:** Clean the columns containing missing values ($NaN$) using imputation strategies:
   * **Categorical Imputation:** Fill high-missingness text columns like `primary_ai_tools_used` with a default placeholder (`'None'`).
   * **Numerical Imputation:** Fill missing numerical features with their **Median** values to keep our distributions stable and unaffected by outliers.

In [12]:
df.isna().sum()  # You can also use df.isnull().sum() — both do the same thing

student_id                            0
age                                   0
gender                                0
degree_type                           0
stream                                0
year_of_study                         0
college_tier                          0
urban_or_rural                        0
daily_ai_tool_usage_hrs               0
primary_ai_tools_used              3215
uses_ai_for_assignments               0
ai_replaces_own_thinking_score        0
ai_dependency_score                   0
placement_anxiety_score               0
fear_of_job_loss_to_ai                0
career_clarity_score                  0
internship_experience                 0
weekly_job_application_count          0
resume_confidence_score               0
interview_anxiety_score               0
daily_study_hours                     0
self_learning_hours_per_week        233
skill_development_courses_taken       0
social_media_hrs_per_day            210
sleep_hours                         203


In [13]:
# We use errors='ignore' so the cell can be re-run safely without crashing
df.drop(columns=['student_id'], inplace=True, errors='ignore')
print("🗑️ Dropped 'student_id' column successfully.")

🗑️ Dropped 'student_id' column successfully.


In [14]:
# For 'primary_ai_tools_used', the massive gaps mean the student likely doesn't use tools.
# We fill these with 'None' to preserve the records.
df['primary_ai_tools_used'] = df['primary_ai_tools_used'].fillna('None')

In [15]:
# For 'seeks_career_counseling', we fill missing values with the most common answer (Mode)
counseling_mode = df['seeks_career_counseling'].mode()[0]
df['seeks_career_counseling'] = df['seeks_career_counseling'].fillna(counseling_mode)

In [17]:
# We select the numerical columns that showed missing values in our initial check
missing_num_cols = ['self_learning_hours_per_week', 'social_media_hrs_per_day', 'sleep_hours']

# Loop through each column and replace NaN values with that column's median
for col in missing_num_cols:
    col_median = df[col].median()
    df[col] = df[col].fillna(col_median)

print("🎯 Missing values handled using clinical imputation strategies.")

🎯 Missing values handled using clinical imputation strategies.


In [18]:
# Check the remaining total number of missing entries across the entire dataset
remaining_nans = df.isna().sum().sum()
print(f"🛑 Remaining Missing Values in DataFrame: {remaining_nans}")
print(f"📊 Cleaned Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns.\n")

🛑 Remaining Missing Values in DataFrame: 0
📊 Cleaned Dataset Dimensions: 15000 rows, 29 columns.



In [19]:
# Display a preview of the clean dataset
df.head()

,age,gender,degree_type,stream,year_of_study,college_tier,urban_or_rural,daily_ai_tool_usage_hrs,primary_ai_tools_used,uses_ai_for_assignments,...,daily_study_hours,self_learning_hours_per_week,skill_development_courses_taken,social_media_hrs_per_day,sleep_hours,stress_level,burnout_score,motivation_score,seeks_career_counseling,overall_career_readiness_score
0,25,Male,B.Tech/B.E.,Engineering (Non-CS),3,Tier 3,Rural,0.3,Perplexity,Never,...,3.5,8.7,2,1.5,8.5,1,3,6,0.0,9.09
1,20,Female,B.Tech/B.E.,Engineering (Non-CS),4,Tier 3,Urban,1.9,ChatGPT,Rarely,...,2.4,8.2,3,4.2,8.3,4,5,3,0.0,6.03
2,25,Female,MBA,CS/IT,1,Tier 3,Urban,3.6,Gemini,Sometimes,...,2.3,16.4,1,2.3,7.0,5,9,3,1.0,6.68
3,23,Male,B.Tech/B.E.,CS/IT,1,Tier 1,Urban,4.1,GitHub Copilot,Frequently,...,7.7,15.0,3,2.6,7.2,6,6,6,0.0,7.71
4,22,Female,MBA,CS/IT,1,Tier 1,Urban,3.4,ChatGPT,Sometimes,...,2.4,1.1,2,1.9,8.0,3,5,4,0.0,4.64
